### DATA PREPARATION

In [13]:

! pip install pandas numpy matplotlib seaborn nltk scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [18]:
tickers = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']
data_files = {ticker: pd.read_csv(f'../data/{ticker}.csv') for ticker in tickers}

# Access by ticker
print(data_files['AAPL'].head())
print(data_files['META'].info())

         Date     Close      High       Low      Open      Volume
0  2009-01-02  2.721686  2.730385  2.554037  2.575630   746015200
1  2009-01-05  2.836553  2.884539  2.780469  2.794266  1181608400
2  2009-01-06  2.789767  2.914229  2.770872  2.877641  1289310400
3  2009-01-07  2.729484  2.774170  2.706990  2.753477   753048800
4  2009-01-08  2.780169  2.793666  2.700393  2.712090   673500800
<class 'pandas.DataFrame'>
RangeIndex: 2923 entries, 0 to 2922
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    2923 non-null   str    
 1   Close   2923 non-null   float64
 2   High    2923 non-null   float64
 3   Low     2923 non-null   float64
 4   Open    2923 non-null   float64
 5   Volume  2923 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 137.1 KB
None


In [19]:
for ticker, df in data_files.items():
    print(f"--- {ticker} ---")
    print(df.isnull().sum())
    print()

--- AAPL ---
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

--- AMZN ---
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

--- GOOG ---
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

--- META ---
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

--- NVDA ---
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64



#### There are no missing values within the datasets.

### Compute Technical Indicators with TA-Lib


In [20]:
# Moving Averages
for ticker, df in data_files.items():
    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['SMA_50'] = df['Close'].rolling(50).mean()
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()

    print(ticker, df[['SMA_20', 'SMA_50', 'EMA_20', 'EMA_50']].tail(2))

AAPL           SMA_20      SMA_50      EMA_20      EMA_50
3772  192.362839  184.479567  191.499696  186.962893
3773  192.490633  184.814828  191.426275  187.110575
AMZN           SMA_20    SMA_50      EMA_20      EMA_50
3772  149.531499  142.5694  150.031597  144.642297
3773  149.824000  143.0456  150.213350  144.928482
GOOG           SMA_20      SMA_50      EMA_20      EMA_50
3772  135.628861  134.023641  137.145637  135.137038
3773  135.976979  134.056417  137.414827  135.326650
META           SMA_20      SMA_50      EMA_20      EMA_50
2921  335.537503  325.527023  340.597473  328.757160
2922  336.869788  326.262290  341.663551  329.660459
NVDA          SMA_20     SMA_50     EMA_20     EMA_50
3772  47.908266  46.512367  48.378593  47.145178
3773  48.046003  46.658888  48.485126  47.237414


In [21]:
# Relative Strength Index
for ticker, df in data_files.items():
    delta = df['Close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / avg_loss
    df['RSI_14'] = 100 - (100 / (1 + rs))

    print(ticker, df[['RSI_14']].tail(2))

AAPL          RSI_14
3772  47.920425
3773  40.185234
AMZN          RSI_14
3772  68.786107
3773  62.417575
GOOG          RSI_14
3772  58.289315
3773  63.741198
META          RSI_14
2921  79.726508
2922  70.564254
NVDA          RSI_14
3772  66.371993
3773  62.559209


In [22]:
# Moving Average Convergence Divergence
for ticker, df in data_files.items():
    df['MACD'] = df['Close'].ewm(span=12, adjust=False).mean() - df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_hist'] = df['MACD'] - df['MACD_signal']

    print(ticker, df[['MACD', 'MACD_signal', 'MACD_hist']].tail(2))

AAPL           MACD  MACD_signal  MACD_hist
3772  1.823998     2.640915  -0.816918
3773  1.559539     2.424640  -0.865101
AMZN           MACD  MACD_signal  MACD_hist
3772  2.989395     2.958303   0.031092
3773  2.782032     2.923049  -0.141017
GOOG           MACD  MACD_signal  MACD_hist
3772  1.855168     1.171730   0.683438
3773  1.842820     1.305948   0.536872
META           MACD  MACD_signal  MACD_hist
2921  8.344500     6.318510   2.025990
2922  8.193282     6.693464   1.499818
NVDA           MACD  MACD_signal  MACD_hist
3772  0.692470     0.619564   0.072906
3773  0.697532     0.635158   0.062374
